# **CIFAR-10 Image Classification with CNN**

A `Sequential` Keras CNN trained on a configurable subset of CIFAR-10, with every run
tracked in MLflow (parameters, metrics, the training-curve figure and the saved model).

Runs in Google Colab or locally — the setup cell detects which and picks its output
paths accordingly. All tunables live in the configuration cell; nothing below it needs
editing for a normal run.

In [ ]:
# --- Environment and output paths -------------------------------------------------
# Detects Colab vs local and chooses where the MLflow store, figures and saved models go.
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/gdrive")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/ImageRecognition")
else:
    OUTPUT_DIR = Path(os.environ.get("IMGCLASS_OUTPUT_DIR", "./outputs")).resolve()

FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "model"
for _d in (OUTPUT_DIR, FIGURE_DIR, MODEL_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"Running in Colab: {IN_COLAB}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# MLflow 3.x is required: the 2.x file-store backend used by earlier versions of this
# notebook now raises MlflowException. Colab already provides TensorFlow/Keras, so this
# deliberately does not reinstall them — see requirements.txt for a local environment.
!pip install "mlflow>=3,<4" --quiet

In [ ]:
import urllib.error
import urllib.request

import cv2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import tensorflow as tf
from tensorflow import keras
from keras.datasets import cifar10
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Input, Dense, Conv2D, MaxPooling2D
from keras.layers import Dropout, Flatten, BatchNormalization
from keras.regularizers import l2
from keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

print(f"TensorFlow {tf.__version__} | Keras {keras.__version__}")

In [ ]:
import mlflow

# SQLite backend. The file store (file:///... or ./mlruns) is in maintenance mode in
# MLflow 3.x and raises unless MLFLOW_ALLOW_FILE_STORE=true; SQLite is the supported
# path. Three slashes plus an absolute path gives the required sqlite://// form.
TRACKING_URI = f"sqlite:///{OUTPUT_DIR / 'mlflow.db'}"
EXPERIMENT_NAME = "MLflow_ImgClass2"

print(TRACKING_URI)

In [ ]:
# --- Configuration ----------------------------------------------------------------
# Everything tunable in one place. Passed straight into ImgClassification(**CONFIG).
SAMPLE_IMAGE_URL = (
    "https://raw.githubusercontent.com/FarzadNekouee/Keras-CIFAR10-CNN-Model/"
    "master/truck_sample.png"
)

CONFIG = dict(
    size_train=4500,        # CIFAR-10 training images to keep (of 50 000)
    size_test=500,          # CIFAR-10 test images to keep (of 10 000)
    validation_split=0.1,   # fraction of size_train that Keras holds out during fit
    batch_size=64,
    epochs=11,
    learning_rate=0.0005,
    weight_decay=0.0001,    # L2 penalty on every Conv2D kernel
    random_seed=0,
    sample_url=SAMPLE_IMAGE_URL,
)

CONFIG

In [ ]:
class ImgClassification:
    """CIFAR-10 CNN classifier with MLflow tracking.

    One call to `train()` loads the data, builds and fits the model, then logs the
    parameters, metrics, training-curve figure and saved model to a single MLflow run.
    """

    CLASS_NAMES = (
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck",
    )

    def __init__(self, size_train=4500, size_test=500, validation_split=0.1,
                 batch_size=64, epochs=11, learning_rate=0.0005, weight_decay=0.0001,
                 random_seed=0, sample_url=None, tracking_uri=None,
                 experiment_name=None, figure_dir=None, model_dir=None):
        keras.utils.set_random_seed(random_seed)

        self.size_train = size_train
        self.size_test = size_test
        self.validation_split = validation_split
        self.batch_size = batch_size
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.random_seed = random_seed
        self.sample_url = sample_url or SAMPLE_IMAGE_URL

        self.tracking_uri = tracking_uri or TRACKING_URI
        self.experiment_name = experiment_name or EXPERIMENT_NAME
        self.figure_dir = Path(figure_dir or FIGURE_DIR)
        self.model_dir = Path(model_dir or MODEL_DIR)

        self.class_names = list(self.CLASS_NAMES)
        self.optimizer = Adam(learning_rate=learning_rate)
        self.model = None
        self.history = None

    # --- Data ---------------------------------------------------------------------

    @staticmethod
    def loaddata():
        (X_train, y_train), (X_test, y_test) = cifar10.load_data()
        return X_train, y_train, X_test, y_test

    def Data2input(self):
        """Load CIFAR-10 and keep the configured subset.

        There is no manual train/validation split: `modelfit` passes
        `validation_split` to Keras, which holds out the tail of the training set.
        """
        X_train, y_train, X_test, y_test = self.loaddata()

        self.X_train = X_train[:self.size_train]
        self.y_train = y_train[:self.size_train]

        self.X_test = X_test[:self.size_test]
        self.y_test = y_test[:self.size_test]

        return self.X_train, self.y_train, self.X_test, self.y_test

    @property
    def n_validation(self):
        """Images Keras will hold out of the training set during `fit`."""
        return int(len(self.X_train) * self.validation_split)

    def Data2Use(self, show_samples=True):
        """Report the shapes actually going into training, and preview the images."""
        n_valid = self.n_validation

        print(f"Train Images Shape:       {self.X_train.shape}")
        print(f"Train Labels Shape:       {self.y_train.shape}")

        print(f"\nValidation split:         {self.validation_split:.0%} of train "
              f"({n_valid} images, held out by Keras during fit)")
        print(f"Images actually fitted:   {len(self.X_train) - n_valid}")

        print(f"\nTest Images Shape:        {self.X_test.shape}")
        print(f"Test Labels Shape:        {self.y_test.shape}")

        if not show_samples:
            return None

        fig = plt.figure(figsize=(15, 15))
        for i in range(min(64, len(self.X_train))):
            plt.subplot(8, 8, i + 1)
            plt.xticks([])
            plt.yticks([])
            plt.grid(False)
            plt.imshow(self.X_train[i])
            plt.title(self.class_names[self.y_train[i][0]], fontsize=12)
        plt.show()
        return fig

    def _normalise(self):
        """Standardise inputs using statistics from the fitted portion only.

        Keras' `validation_split` takes the LAST fraction of the training set, so the
        mean and standard deviation come from everything before that slice. Computing
        them over the whole array would leak validation statistics into training.
        """
        n_valid = self.n_validation
        fitted_portion = self.X_train[:-n_valid] if n_valid else self.X_train

        self.mean = float(np.mean(fitted_portion))
        self.std = float(np.std(fitted_portion))

        self.X_train = (self.X_train.astype("float32") - self.mean) / (self.std + 1e-7)
        self.X_test = (self.X_test.astype("float32") - self.mean) / (self.std + 1e-7)

        self.y_train = to_categorical(self.y_train, 10)
        self.y_test_categorical = to_categorical(self.y_test, 10)

    # --- Model --------------------------------------------------------------------

    def ClassCNN(self):
        """Normalise the inputs, then build and compile the CNN."""
        self._normalise()

        weight_decay = self.weight_decay

        self.model = Sequential()
        self.model.add(Input(shape=self.X_train.shape[1:]))

        self.model.add(Conv2D(filters=32, kernel_size=(3, 3), padding="same", activation="relu", kernel_regularizer=l2(weight_decay)))
        self.model.add(BatchNormalization())
        self.model.add(Conv2D(filters=32, kernel_size=(3, 3), padding="same", activation="relu", kernel_regularizer=l2(weight_decay)))
        self.model.add(BatchNormalization())

        self.model.add(MaxPooling2D(pool_size=(2, 2)))
        self.model.add(Dropout(rate=0.2))

        self.model.add(Conv2D(filters=64, kernel_size=(3, 3), padding="same", activation="relu", kernel_regularizer=l2(weight_decay)))
        self.model.add(BatchNormalization())
        self.model.add(Conv2D(filters=64, kernel_size=(3, 3), padding="same", activation="relu", kernel_regularizer=l2(weight_decay)))
        self.model.add(BatchNormalization())

        self.model.add(MaxPooling2D(pool_size=(2, 2)))
        self.model.add(Dropout(rate=0.3))

        self.model.add(Conv2D(filters=128, kernel_size=(3, 3), padding="same", activation="relu", kernel_regularizer=l2(weight_decay)))
        self.model.add(BatchNormalization())
        self.model.add(Conv2D(filters=128, kernel_size=(3, 3), padding="same", activation="relu", kernel_regularizer=l2(weight_decay)))
        self.model.add(BatchNormalization())

        self.model.add(MaxPooling2D(pool_size=(2, 2)))
        self.model.add(Dropout(rate=0.4))

        self.model.add(Conv2D(filters=256, kernel_size=(3, 3), padding="same", activation="relu", kernel_regularizer=l2(weight_decay)))
        self.model.add(BatchNormalization())
        self.model.add(Conv2D(filters=256, kernel_size=(3, 3), padding="same", activation="relu", kernel_regularizer=l2(weight_decay)))
        self.model.add(BatchNormalization())

        self.model.add(MaxPooling2D(pool_size=(2, 2)))
        self.model.add(Dropout(rate=0.5))
        self.model.add(Flatten())
        self.model.add(Dense(10, activation="softmax"))

        self.model.summary()

        self.model.compile(optimizer=self.optimizer,
                           loss="categorical_crossentropy",
                           metrics=["accuracy"])

        # Patience is scaled to the epoch budget. The original values (10 and 40) could
        # never fire inside an 11-epoch run, so neither callback did anything.
        self.reduce_lr = ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=max(2, self.epochs // 4), min_lr=1e-5, verbose=1,
        )
        self.early_stopping = EarlyStopping(
            monitor="val_loss", patience=max(3, self.epochs // 2),
            restore_best_weights=True, verbose=1,
        )

        return self.model

    def modelfit(self):
        """Fit the model, holding out `validation_split` of the training set."""
        self.history = self.model.fit(
            self.X_train, self.y_train,
            batch_size=self.batch_size,
            epochs=self.epochs,
            validation_split=self.validation_split,
            callbacks=[self.reduce_lr, self.early_stopping],
            verbose=2,
        )
        return self.history

    # --- Reporting ----------------------------------------------------------------

    def plots(self):
        """Loss and accuracy curves. Saved to disk and logged to the active run."""
        history = self.history.history

        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].plot(history["loss"], label="Train Loss", color="#8502d1")
        ax[0].plot(history["val_loss"], label="Validation Loss", color="darkorange")
        ax[0].legend()
        ax[0].set_title("Loss Evolution")
        ax[0].set_xlabel("Epoch")

        ax[1].plot(history["accuracy"], label="Train Accuracy", color="#8502d1")
        ax[1].plot(history["val_accuracy"], label="Validation Accuracy", color="darkorange")
        ax[1].legend()
        ax[1].set_title("Accuracy Evolution")
        ax[1].set_xlabel("Epoch")
        fig.tight_layout()

        # Save and log BEFORE plt.show(): show() closes the figure, so the original
        # savefig-after-show wrote an empty image.
        figure_path = self.figure_dir / "generatedCIFAR_images_accuracy.png"
        fig.savefig(figure_path, dpi=150)
        mlflow.log_figure(fig, artifact_file="plot/GenImage.png")
        plt.show()

        print(f"Figure written to {figure_path}")
        return figure_path

    def evaluate(self):
        """Score the held-out test set and log every metric to the active run."""
        test_loss, test_acc = self.model.evaluate(
            self.X_test, self.y_test_categorical, verbose=0
        )

        y_prob = self.model.predict(self.X_test, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)
        y_true = self.y_test.ravel()

        metrics = {
            "test_loss": float(test_loss),
            "test_accuracy": float(test_acc),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        }
        mlflow.log_metrics(metrics)

        print(f"\nTest Accuracy: {metrics['test_accuracy']:.4f}")
        print(f"Test Loss:     {metrics['test_loss']:.4f}")
        print(f"Weighted F1:   {metrics['f1']:.4f}")
        return metrics

    def predict_sample(self):
        """Classify the reference truck image.

        Skipped with a warning if the download fails, so a network blip cannot abort
        an otherwise complete run after training has finished.
        """
        try:
            with urllib.request.urlopen(self.sample_url, timeout=30) as resp:
                buffer = np.asarray(bytearray(resp.read()), dtype="uint8")
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            print(f"Sample image skipped ({exc})")
            mlflow.set_tag("sample_prediction", "skipped")
            return None

        image = cv2.imdecode(buffer, cv2.IMREAD_UNCHANGED)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        plt.imshow(image)
        plt.xticks([])
        plt.yticks([])
        plt.grid(False)
        plt.show()

        image = cv2.resize(image, (32, 32))
        image = (image.astype("float32") - self.mean) / (self.std + 1e-7)
        image = image.reshape((1, 32, 32, 3))

        predicted_class = self.class_names[self.model.predict(image, verbose=0).argmax()]

        print(f"Predicted class: {predicted_class}")
        mlflow.set_tag("sample_prediction", predicted_class)
        return predicted_class

    # --- MLflow -------------------------------------------------------------------

    def runMLflow(self):
        """Point MLflow at the SQLite store and select the experiment."""
        mlflow.set_tracking_uri(self.tracking_uri)
        mlflow.set_experiment(self.experiment_name)
        print(f"Tracking to {self.tracking_uri} / {self.experiment_name}")

    def _log_params(self):
        mlflow.log_params({
            "size_train": self.size_train,
            "size_test": self.size_test,
            "validation_split": self.validation_split,
            "batch_size": self.batch_size,
            "epochs": self.epochs,
            "optimizer": type(self.optimizer).__name__,
            "learning_rate": self.learning_rate,
            "weight_decay": self.weight_decay,
            "random_seed": self.random_seed,
        })
        mlflow.set_tag("dataset", "CIFAR-10")

    def _log_model(self):
        """Save the trained model and attach it to the run.

        Written as a `.keras` file and logged as an artifact rather than through
        `mlflow.keras.log_model`, whose signature has changed across MLflow releases.
        """
        model_path = self.model_dir / "cifar10_cnn.keras"
        self.model.save(model_path)
        mlflow.log_artifact(str(model_path), artifact_path="model")
        print(f"Model written to {model_path}")
        return model_path

    def computemetrics(self):
        """Run everything that belongs to one tracked MLflow run."""
        with mlflow.start_run() as run:
            self._log_params()

            self.ClassCNN()
            self.modelfit()

            self.plots()
            metrics = self.evaluate()
            self.predict_sample()
            self._log_model()

            print(f"\nMLflow run: {run.info.run_id}")
            return run.info.run_id, metrics

    # --- Entry point --------------------------------------------------------------

    def train(self, show_samples=True):
        """Load the data, fit the model and log one complete MLflow run."""
        self.Data2input()
        self.Data2Use(show_samples=show_samples)
        self.runMLflow()
        return self.computemetrics()

In [ ]:
classifier = ImgClassification(**CONFIG)
run_id, metrics = classifier.train()

## Viewing the tracked runs

The runs live in a SQLite database next to this notebook's output directory. To browse
them, point the MLflow UI at the same URI:

```bash
mlflow ui --backend-store-uri sqlite:///outputs/mlflow.db
```

On Colab the database is on Drive at `MyDrive/ImageRecognition/mlflow.db` — copy it
down (together with the `mlartifacts` folder beside it) and run the command above
locally.

In [ ]:
# Inspect the logged runs without leaving the notebook.
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
columns = [c for c in ("run_id", "start_time", "metrics.test_accuracy", "metrics.f1",
                       "params.size_train", "params.epochs") if c in runs.columns]
runs[columns].head(10)